# Disjoint-block feature set -- does it fix the grind-mode barcode?

Two tied observations drove this notebook:

1. **User, reading the regime-on-price charts**: the detector reads violent
   moves well and "barcodes" through sustained grinds -- rapid colour flips
   where a human sees one coherent trend.
2. **User, on the feature set**: "isn't the parameters too many? ... the
   noise could be too much." Measured: the shipped 9 features carry only
   4.20 effective dimensions (participation ratio); `mom_3d`/`mom_5d`/
   `drawdown`/`dist_ma` correlate at 0.71-0.88 -- one dimension measured four
   ways, with a 5-day reach.

Full reasoning, the earlier (wrong) "just add a 60-day rung" idea and why it
was retracted, and the pre-registered predictions: `FX_FEATURESET_PROTOCOL.md`.

**This is not a parameter search.** The block edges (1/5/20/60 days) and the
dropped feature were fixed by measured collinearity structure BEFORE this
notebook ran on any instrument. All 7 instruments are graded together.

In [ ]:
%pip install -q hmmlearn

## 1. Engine, constants, metrics, data layer — harvested verbatim

From `build_intraday_fx.py` cells 1, [3, 4, 5, 6, 7], [9, 10, 11, 12, 13, 14], [18, 19, 20], [22], 28 (definitions only). `build_features`, `FEATURE_COLS`, `FEATURE_SIGN`, `FEATURE_MAG`, `DIRECTION_EXCLUDE`, `BAR_DIR_FEATURES` and `TREND_FEATURE` are the shipped (NESTED) values at this point -- untouched.

In [ ]:
# ==========================================================================
# CONSTANTS + IMPORTS -- INLINED VERBATIM from build_master_notebook_v2.py
# (master notebook code cell 3). Unmodified.
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier. Windows are now set
                               # explicitly in BASE_WIN (see below), so this stays 1.0.
# TIMESCALE NOTE (kept: this is the measurement the window choice rests on).
# Originally the LONGEST window in the whole feature
# set was SWING_WIN = 20 bars = 6.7 trading days, and TREND_FEATURE (mom_3d) saw
# 3 days. A multi-week decline therefore could not be perceived as one structure:
# inside any 6.7-day slice of it there genuinely ARE bullish stretches, so the
# detector printed bull bars part-way down a sustained selloff. That is a
# TIMESCALE MISMATCH, not a labeling bug -- the engine was answering a shorter
# question than the chart was being judged on.
# MEASURED (real daily NIFTY, same fit config, only the reach changed; bars inside
# a sustained decline = 20-bar return <= -8%):
#     reach ~7 days  -> bear 90.7%   bull 7.4%   side 1.9%
#     reach 20 days  -> bear 100.0%  bull 0.0%   side 0.0%
#     reach 60 days  -> bear 100.0%  bull 0.0%   side 0.0%   (no further gain)
# COST of a longer reach: slower reaction. Inherent, not a defect.
# The windows chosen below (momentum 5d, context 12d) sit BETWEEN the measured
# 7-day and 20-day points, so expect PART of the improvement above, not all of it.
# Any window change CHANGES EMITTED LABELS -- a configuration decision, not a bug fix.

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = max(2, int(round(9 * LOOKBACK_SCALE)))
                        # bars over which efficiency is measured (~ TREND_FEATURE horizon).
                        # NOW SCALES WITH LOOKBACK_SCALE. It previously did not, because
                        # it is a standalone constant rather than a BASE_WIN entry -- so
                        # lengthening the features left the chop/efficiency window at 9
                        # bars (3 days). That would pair a 16-day DIRECTION axis with a
                        # 3-day INTENSITY axis and produce H/L flicker inside an otherwise
                        # stable regime. At LOOKBACK_SCALE=1.0 this is exactly 9, so the
                        # original behaviour is reproduced bit-for-bit.
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.5 -- the v6 configuration. Fix 4 is ON.
#
# ON THE RECORD, REPORTED AND NOT ACTED ON HERE. Scored across three real-data runs,
# the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# That evidence is about w=0.75 vs w=0.0. This build ships v6's w=0.5 and does NOT
# adopt either endpoint. Section 7W re-measures the whole grid, under protocol,
# on whatever data this run actually loaded, and ships none of its arms.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.5
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=4 is the v6 value and
# is what ships here. A later offline study measured direction-call agreement
# between independent pools at 81.9% (single) -> 91.9% (K=6); that is reported,
# not adopted -- K is a configuration choice, not a defect.
ENSEMBLE_K     = 4
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

# WINDOW LENGTHS, in bars. Stated in CALENDAR DAYS x BARS_PER_DAY so the intended
# horizon is readable rather than buried in a bar count.
#   momentum ladder : 1d / 3d / 5d   -- the direction/trend signal, deliberately fast
#   context window  : 12d            -- drawdown + dist_ma, the STRUCTURE the detector
#                                       is judged against. Was 20 bars = 6.7d, which
#                                       is why a multi-week decline could not be seen
#                                       as one structure (see the TIMESCALE NOTE above).
# Split deliberately: a fast momentum ladder keeps the detector responsive, while a
# longer context window gives it something to be responsive WITHIN. Scaling every
# window together (the old uniform LOOKBACK_SCALE route) buys the context at the
# cost of the responsiveness.
CONTEXT_DAYS = 12
BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5,
                VOL_SLOW=CONTEXT_DAYS*BARS_PER_DAY,
                SWING_WIN=CONTEXT_DAYS*BARS_PER_DAY)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'frozen_z'    # 'frozen_z' = v6 / pre-change | 'vol_norm' = post-v6 option
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'allow'   # 'allow' (v6 / pre-change) | 'block' | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON at w={BAR_DIR_WEIGHT}: direction is "
          f"{100*(1-BAR_DIR_WEIGHT):.0f}% per-STATE (HMM) + "
          f"{100*BAR_DIR_WEIGHT:.0f}% per-BAR.")
    print("              Swept end to end in 7H-vi and under protocol in 7W; "
          "neither ships an arm.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# DATA LOADER -- INLINED VERBATIM (master code cell 5).
# Supplies TAC_SYNTH and _synth(). Its Nifty series is NOT used below.
# ==========================================================================
def load_2h():
    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""
    try:
        import yfinance as yf

        def h2(tk):
            raw = yf.download(tk, interval='60m', period='730d',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index
            return c.resample('2h').last().dropna()

        nifty = h2('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = h2('^INDIAVIX')
            # BUGFIX (look-ahead): this was `.ffill().bfill()`. After ffill the only
            # remaining NaNs are LEADING ones -- bars before VIX's first print --
            # and bfill filled those from the FIRST FUTURE observation. That is a
            # look-ahead: bar t took a value that did not exist until t+k.
            # It is invisible to the truncation probe, because the leak is anchored
            # to the START of the series, not to the cut point.
            # ffill only; the leading NaNs then fall out of build_features' dropna(),
            # which is correct -- those bars genuinely carry no VIX information.
            if vix is not None and len(vix):
                vix = vix.reindex(nifty.index).ffill()
                _lead = int(vix.isna().sum())
                if _lead:
                    print(f'  VIX starts after NIFTY: {_lead} leading bars have no VIX '
                          f'and will be dropped by the feature warmup (previously these '
                          f'were back-filled from the future).')
            else:
                vix = pd.Series(15.0, index=nifty.index)
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'yfinance 60m->2h: {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few bars')
    except Exception as e:
        print(f'yfinance unavailable ({e}); falling back to synthetic 2h data.')
        return _synth()


def _synth():
    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),
           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),
           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),
           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]
    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []
    for i in range(n):
        f = i / n
        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, q in REG:
            if fs <= f < fe:
                mu, sg, vm, vf = m, s, v, q
                break
        lvl *= np.exp(rng.normal(mu, sg))
        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nv.append(lvl); vv.append(pv)
    return (pd.Series(nv, index=idx, name='nifty'),
            pd.Series(vv, index=idx, name='vix'), True)


nifty, vix, TAC_SYNTH = load_2h()
if TAC_SYNTH:
    print(f"\n*** TAC_SYNTH = True  ->  SYNTHETIC data ({len(nifty)} bars). "
          f"Results below are ILLUSTRATIVE ONLY. ***\n")
else:
    print(f"\n*** TAC_SYNTH = False  ->  REAL yfinance data ({len(nifty)} bars). "
          f"Results below are decision-grade. ***\n")
print(f"Span: {nifty.index[0]}  ->  {nifty.index[-1]}")

In [ ]:
# ==========================================================================
# FEATURE + LABELING ENGINE -- INLINED VERBATIM (master code cell 7).
# build_features / fit_hmm_ensemble / confirm_delay / label_bars / n_params.
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    # HYSTERESIS INVARIANT: the exit band must sit at or inside the enter band.
    # If exit were STRICTER than enter, a bar could fail the exit test (state -> 0)
    # and then immediately clear the enter test on the SAME bar, silently turning
    # the hysteresis into a no-op. Equality is allowed and is used deliberately:
    # z_exit == z_hi and eff_exit == eff_hi reduce this to the memoryless test,
    # which the equivalence checks rely on.
    assert z_exit <= z_hi, f'z_exit {z_exit} must be <= z_hi {z_hi} (hysteresis inverted)'
    assert eff_exit <= eff_hi, f'eff_exit {eff_exit} must be <= eff_hi {eff_hi} (hysteresis inverted)'
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    # BUGFIX (symmetry): the old form was
    #     n_side = max(1, round(n_st / 5)); n_bear = (n_st - n_side) // 2
    # which floor-divides the remainder and hands EVERY leftover state to the BULL
    # bucket -- a structural bullish tilt with no basis in the data. Measured:
    # N=4 -> 1/1/2, N=6 -> 2/1/3, N=9 -> 3/2/4 (bear/side/bull). N=6 matters
    # because the capacity study recommended it.
    # Now: bear and bull are EQUAL by construction and any remainder is absorbed
    # by the SIDE bucket, which is the "undecided" bucket and the only one where an
    # unassignable state belongs. N=3,5,7,8,10 are UNCHANGED by this -- in
    # particular the shipped N=5 stays 2/1/2 bit-for-bit.
    n_bear = (n_st - max(1, round(n_st / 5))) // 2
    n_side = n_st - 2 * n_bear                 # absorbs the remainder, keeps symmetry
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    # INVARIANT: no directional tilt may be introduced by the bucketing itself.
    assert (direction == 1).sum() == (direction == -1).sum(), \
        f'bull/bear buckets asymmetric at N={n_st} -> structural directional bias'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# PLOT HELPERS -- INLINED VERBATIM (master code cell 9).
# regime_blocks / shade_bands / shade_regimes / set_price_ylim /
# regime_legend / mark_split / YLIM_CHECKS.
# This is the MASTER NOTEBOOK regime-background style (full-height bands,
# each block extended to the START of the next so there are no white stripes,
# explicit non-zero-anchored y-limits).
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    # BUGFIX (rendering): a block used to end at idx[i-1] -- its own LAST bar --
    # while the next block began at idx[i]. The interval between them was left
    # unpainted. Within a session that gap is one bar (2h) and invisible; across
    # an overnight it is ~18h and across a weekend ~64h, so on a multi-year chart
    # with 500+ blocks the background broke up into white stripes wherever a
    # regime boundary happened to fall on non-trading time. That made the regimes
    # look far more fragmented than they are -- on the exact figure used to judge
    # versions by eye. Each block now runs to the START of the next one, so the
    # shading is gap-free and every bar's colour still spans its own bar.
    # Plotting only: `regime_blocks` has exactly one caller, `shade_bands`, so no
    # occupancy, run-length or label statistic anywhere depends on these edges.
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i]))   # -> next block's start
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    # INVARIANT: consecutive blocks must abut exactly -- no unpainted slivers.
    for _a, _b in zip(blocks, blocks[1:]):
        assert _a[2] == _b[1], 'regime shading spans must abut (unpainted gap)'
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

In [ ]:
# ==========================================================================
# HAC (Newey-West) CONTRAST -- INLINED VERBATIM from the master generator
# (the two function definitions only; the sweep that surrounds them is not
# run here). This is the SAME estimator the master, the w-sweep and
# generalization.ipynb use, so every t-stat here is comparable with theirs.
# ==========================================================================
def _w_hac_contrast(y, grp, lag):
    # HAC (Newey-West, Bartlett) t-stat for the CONTRAST (mean of group +1)
    # minus (mean of group -1), estimated as an OLS with a SATURATED dummy
    # design over the FULL time-ordered span.
    #
    # WHY NOT SUBSET FIRST. Taking y[bull] and y[bear] as two separate vectors
    # and running Newey-West on each destroys the time index: bars that were 40
    # apart become adjacent, and the Bartlett kernel then weights a lag that is
    # not the lag it thinks it is. Keeping every bar in the design (group 0 =
    # everything else) preserves the real spacing, and with orthogonal indicator
    # columns and no intercept the coefficients ARE the group means exactly, so
    # beta[+1] - beta[-1] is the difference of means with nothing approximated.
    #
    # Returns (diff, se, t, n_pos, n_neg).
    y = np.asarray(y, dtype=float)
    grp = np.asarray(grp, dtype=float)
    ok = np.isfinite(y)
    y = y[ok]
    grp = grp[ok]
    n = len(y)
    if n < 10:
        return np.nan, np.nan, np.nan, 0, 0
    cols = [(grp > 0).astype(float), (grp < 0).astype(float), (grp == 0).astype(float)]
    keep = [j for j, c in enumerate(cols) if c.sum() > 0]
    if 0 not in keep or 1 not in keep:
        return np.nan, np.nan, np.nan, int(cols[0].sum()), int(cols[1].sum())
    X = np.column_stack([cols[j] for j in keep])
    XtX = X.T @ X
    XtXi = np.linalg.pinv(XtX)
    beta = XtXi @ (X.T @ y)
    u = y - X @ beta
    Z = X * u[:, None]                       # score contributions x_t * u_t
    lag = max(0, min(int(lag), n - 1))
    S = Z.T @ Z
    for k in range(1, lag + 1):
        wk = 1.0 - k / (lag + 1.0)           # Bartlett weight
        G = Z[k:].T @ Z[:-k]
        S = S + wk * (G + G.T)
    V = XtXi @ S @ XtXi
    c = np.zeros(len(keep))
    c[keep.index(0)] = 1.0
    c[keep.index(1)] = -1.0
    diff = float(c @ beta)
    var = float(c @ V @ c)
    se = np.sqrt(var) if var > 0 else np.nan
    t = diff / se if (se and np.isfinite(se) and se > 0) else np.nan
    return diff, se, t, int(cols[0].sum()), int(cols[1].sum())


def W_PRIMARY_METRIC(labels, close_arr, horizons=None):
    # THE FROZEN PRIMARY CRITERION (W_SWEEP_PROTOCOL.md).
    #
    #   PRIMARY = mean over h of the HAC t-stat of
    #             (mean fwd_h | BULL bars) - (mean fwd_h | BEAR bars)
    #
    # Rationale (protocol, verbatim in substance): this is the one quantity the
    # detector exists to produce -- a directional split that survives overlap
    # correction. It is not a composite of unrelated families and it cannot be
    # gamed by occupancy.
    #
    # Returns (primary_scalar, per_horizon_dict). NaN-safe.
    horizons = list(FWD_HORIZONS if horizons is None else horizons)
    labels = np.asarray(labels, dtype=object)
    close_arr = np.asarray(close_arr, dtype=float)
    grp = np.where(np.isin(labels, ['H_BULL', 'L_BULL']), 1.0,
                   np.where(np.isin(labels, ['H_BEAR', 'L_BEAR']), -1.0, 0.0))
    per_h = {}
    ts = []
    for h in horizons:
        f = np.full(len(close_arr), np.nan)
        if h < len(close_arr):
            f[:len(close_arr) - h] = (close_arr[h:] / close_arr[:len(close_arr) - h] - 1.0) * 100.0
        d, se, t, npos, nneg = _w_hac_contrast(f, grp, lag=h - 1)
        per_h[h] = {'diff_pct': d, 'hac_se': se, 'hac_t': t, 'n_bull': npos, 'n_bear': nneg}
        ts.append(t)
    ts = [t for t in ts if np.isfinite(t)]
    return (float(np.mean(ts)) if ts else np.nan), per_h

In [ ]:
# ===========================================================================
# METRIC + GRID PARAMETERS -- ALL DECLARED HERE, BEFORE ANY NUMBER EXISTS.
# Values are IDENTICAL to build_stability_lag.py and build_generalization.py so
# the three notebooks compare directly.
# ===========================================================================
import itertools, time, contextlib, io, sys, urllib.request
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap

T_START = time.time()

R_SEED_SETS   = 4       # protocol: R = 4 DISJOINT seed sets. NEVER cut.
ZZ_PCT        = 2.0     # ZigZag reversal threshold, %
W_GUARD_MAX   = 25.0    # G3
OCC_MIN_PCT   = 3.0     # G1 / G2
OCC_MAX_PCT   = 50.0    # G1
SIDEWAYS_MAX  = 50.0    # G4

# Pre-registered P2 band (protocol text: "roughly 20-45%").
P2_LO, P2_HI  = 20.0, 45.0

# Day-denominated windows. Same day counts as the master; only the bars-per-day
# multiplier moves.
MOM_DAYS      = (1, 3, 5)
FWD_DAYS      = (1, 3, 5)

print(f'R = {R_SEED_SETS} disjoint seed sets   ZigZag = {ZZ_PCT}%   '
      f'G3 W<={W_GUARD_MAX}   G1 occ in [{OCC_MIN_PCT},{OCC_MAX_PCT}]%   '
      f'G4 SIDEWAYS<={SIDEWAYS_MAX}%')
print(f'momentum ladder {MOM_DAYS} days   CONTEXT_DAYS={CONTEXT_DAYS}   '
      f'forward horizons {FWD_DAYS} days')
print('All declared before a single run exists.')

In [ ]:
# ==========================================================================
# ZIGZAG -- INLINED VERBATIM from build_stability_lag.py.
# Label-blind, retrospective. Registers itself in ZZ_CALLS/ZZ_IDS/ZZ_ARRAYS
# so the leakage tripwire below can prove it never reached a label.
# That generator ships its own tripwire in the SAME cell, hard-coded to
# bar_dir_weight == 0.0; it is cut off here because this notebook installs
# the same four checks anchored to the frozen BAR_DIR_WEIGHT (0.5) instead.
# ==========================================================================
# ===========================================================================
# THE ZIGZAG -- LABEL-BLIND, RETROSPECTIVE. Sees prices only; never a label.
# ===========================================================================
ZZ_CALLS = 0            # how many times a ZigZag has been computed
ZZ_IDS = set()          # id() of every object a ZigZag has ever returned
ZZ_ARRAYS = []          # the arrays themselves, for np.shares_memory checks


def zigzag_pivots(px, pct=ZZ_PCT):
    '''Indices of ZigZag pivots on a close series, `pct`% reversal threshold.

    RETROSPECTIVE BY CONSTRUCTION: a pivot at bar i is only confirmed once price
    has moved pct% away from it, which happens at some bar > i. That is exactly
    the look-ahead this function is allowed to have and `label_bars` is not.

    Takes a bare float array in and returns a bare int array out -- it has no
    access to a label, a mass, a model or a feature frame.
    '''
    global ZZ_CALLS
    px = np.asarray(px, dtype=float)
    n = len(px)
    if n < 3:
        out = np.array([], dtype=int)
    else:
        hi_i = lo_i = 0
        d, start, piv, ext_i = 0, None, [], 0
        for i in range(1, n):
            if px[i] > px[hi_i]:
                hi_i = i
            if px[i] < px[lo_i]:
                lo_i = i
            if (px[hi_i] - px[i]) / px[hi_i] * 100.0 >= pct:
                d, piv, ext_i, start = -1, [hi_i], i, i
                break
            if (px[i] - px[lo_i]) / px[lo_i] * 100.0 >= pct:
                d, piv, ext_i, start = +1, [lo_i], i, i
                break
        if d == 0:
            out = np.array([], dtype=int)
        else:
            for i in range(start + 1, n):
                if d > 0:
                    if px[i] >= px[ext_i]:
                        ext_i = i
                    elif (px[ext_i] - px[i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = -1, i
                else:
                    if px[i] <= px[ext_i]:
                        ext_i = i
                    elif (px[i] - px[ext_i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = +1, i
            if ext_i != piv[-1]:
                piv.append(ext_i)          # final, PROVISIONAL pivot
            out = np.asarray(piv, dtype=int)

    ZZ_CALLS += 1
    ZZ_IDS.add(id(out))
    ZZ_ARRAYS.append(out)
    return out


def zigzag_swings(px, pct=ZZ_PCT):
    '''[(i0, i1, +1/-1), ...] -- consecutive pivot pairs that clear `pct`.

    The last (provisional) leg is kept only if it actually cleared the threshold,
    so a half-formed leg at the right edge cannot inflate or deflate lag.
    '''
    px = np.asarray(px, dtype=float)
    piv = zigzag_pivots(px, pct)
    sw = []
    for a, b in zip(piv[:-1], piv[1:]):
        move = (px[b] - px[a]) / px[a] * 100.0
        if abs(move) >= pct:
            sw.append((int(a), int(b), 1 if move > 0 else -1))
    out = sw
    ZZ_IDS.add(id(out))
    return out

In [ ]:
# ==========================================================================
# S / L / W + GUARDS G1-G4 -- INLINED VERBATIM from build_stability_lag.py.
# Identical definitions to the stability/lag and generalization notebooks,
# so the numbers in all three are directly comparable.
# ==========================================================================
# ===========================================================================
# S / L / W  -- the three metrics, one function each. No composite anywhere.
# ===========================================================================
DIR_OF_LABEL = {'H_BULL': 1, 'L_BULL': 1, 'SIDEWAYS': 0, 'L_BEAR': -1, 'H_BEAR': -1}


def emitted_direction(labels):
    '''5-label string array -> +1 bull / 0 sideways / -1 bear.'''
    return np.array([DIR_OF_LABEL[x] for x in np.asarray(labels)], dtype=int)


def metric_S(label_sets):
    '''S = mean pairwise 5-LABEL agreement (%) across R seed sets.

    Returns (S_mean, pairwise_matrix RxR, list_of_pairwise_values).
    The FULL matrix is returned because the protocol requires it reported, not
    just the mean.
    '''
    R = len(label_sets)
    M = np.full((R, R), np.nan)
    vals = []
    for i in range(R):
        M[i, i] = 100.0
        for j in range(i + 1, R):
            a = np.asarray(label_sets[i]); b = np.asarray(label_sets[j])
            assert len(a) == len(b), 'label sets must be the same length'
            v = 100.0 * float(np.mean(a == b))
            M[i, j] = M[j, i] = v
            vals.append(v)
    return float(np.mean(vals)), M, vals


def metric_L(labels, swings):
    '''L per ZigZag swing = bars from swing START to the first correctly
    directed EMITTED label inside the swing.

    An UNMATCHED swing (never labelled correctly anywhere inside it) scores the
    FULL swing length -- the worst case, per protocol. It is not dropped.

    Returns (median, p75, per_swing_array, n_unmatched).
    '''
    d = emitted_direction(labels)
    lags, unmatched = [], 0
    for i0, i1, sgn in swings:
        seg = d[i0:i1 + 1]
        hit = np.flatnonzero(seg == sgn)
        if hit.size:
            lags.append(int(hit[0]))
        else:
            lags.append(int(i1 - i0))
            unmatched += 1
    if not lags:
        return np.nan, np.nan, np.array([]), 0
    a = np.asarray(lags, dtype=float)
    return float(np.median(a)), float(np.percentile(a, 75)), a, unmatched


def metric_W(labels):
    '''W = 5-label switches per 100 bars.'''
    a = np.asarray(labels)
    if len(a) < 2:
        return 0.0
    return 100.0 * float(np.sum(a[1:] != a[:-1])) / (len(a) - 1)


def occupancy_pct(labels):
    a = np.asarray(labels)
    return {lb: 100.0 * float(np.mean(a == lb)) for lb in REGIME_LABELS}


# ---------------------------------------------------------------------------
# GUARDS G1..G4. One implementation, used by real arms AND by the degeneracy
# stub in section 3, so the test exercises the shipping code path.
# ---------------------------------------------------------------------------
def evaluate_guards(occ, W):
    '''{name: (bool_pass, reason)} for the four protocol guards.'''
    g = {}
    bad_hi = [l for l in REGIME_LABELS if occ.get(l, 0.0) > OCC_MAX_PCT]
    bad_lo = [l for l in REGIME_LABELS if occ.get(l, 0.0) < OCC_MIN_PCT]
    g['G1'] = (not bad_hi and not bad_lo,
               'ok' if not (bad_hi or bad_lo)
               else f'outside [{OCC_MIN_PCT},{OCC_MAX_PCT}]%: '
                    + ','.join(sorted(set(bad_hi + bad_lo))))
    g['G2'] = (not bad_lo, 'ok' if not bad_lo else 'collapsed: ' + ','.join(bad_lo))
    g['G3'] = (W <= W_GUARD_MAX, f'W={W:.2f} vs max {W_GUARD_MAX}')
    g['G4'] = (occ.get('SIDEWAYS', 0.0) <= SIDEWAYS_MAX,
               f"SIDEWAYS={occ.get('SIDEWAYS', 0.0):.1f}% vs max {SIDEWAYS_MAX}%")
    return g


def guards_pass(g):
    return all(v[0] for v in g.values())


print('metrics ready: metric_S (pairwise matrix), metric_L (median/p75, unmatched '
      '= full swing length), metric_W, evaluate_guards (G1-G4).')
print('NO composite score is defined anywhere in this notebook -- by design.')

In [ ]:
# ===========================================================================
# THE FIXED-KNOB INVARIANT.
# Every constant GENERALIZATION_PROTOCOL.md forbids re-tuning is captured here
# and re-asserted after EVERY run. If any of them moves, the whole test is void
# and this assert says so.
# ===========================================================================
FIXED_KNOBS = dict(
    N_STATES               = 5,
    COVARIANCE             = 'diag',
    N_FEATURES             = len(FEATURE_COLS),
    FEATURE_COLS           = tuple(FEATURE_COLS),
    CONF_L                 = CONF_L,
    CONFIRM_BARS           = CONFIRM_BARS,
    BAR_DIR_WEIGHT         = BAR_DIR_WEIGHT,
    ENSEMBLE_K             = ENSEMBLE_K,
    Z_HI                   = Z_HI,
    EFF_HI                 = EFF_HI,
    Z_HI_EXIT              = Z_HI_EXIT,
    EFF_HI_EXIT            = EFF_HI_EXIT,
    EFF_WIN                = EFF_WIN,
    MOM_DAYS               = MOM_DAYS,
    CONTEXT_DAYS           = CONTEXT_DAYS,
    INTENSITY_MODE         = INTENSITY_MODE,
    ESCALATION_DURING_HOLD = ESCALATION_DURING_HOLD,
    DIRECTION_MODE         = DIRECTION_MODE,
    TRAIN_FRACTION         = TRAIN_FRACTION,
    N_FOLDS                = N_FOLDS,
    DIRECTION_EXCLUDE      = tuple(DIRECTION_EXCLUDE),
    H_TARGET_RATE          = H_TARGET_RATE,
    H_EXIT_SLACK           = H_EXIT_SLACK,
    BASE_SEED              = BASE_SEED,
)

# The protocol's literal values, hard-coded, so this notebook fails loudly if
# the harvested master ever drifts away from the frozen spec.
PROTOCOL_VALUES = dict(
    N_STATES=5, COVARIANCE='diag', N_FEATURES=9, CONF_L=0.50, CONFIRM_BARS=2,
    BAR_DIR_WEIGHT=0.5, ENSEMBLE_K=4, Z_HI=0.5, EFF_HI=0.35, Z_HI_EXIT=0.35,
    EFF_HI_EXIT=0.25, EFF_WIN=9, CONTEXT_DAYS=12, INTENSITY_MODE='frozen_z',
    ESCALATION_DURING_HOLD='allow', DIRECTION_MODE='rank',
    TRAIN_FRACTION=0.70, N_FOLDS=4,
)
_bad = {k: (FIXED_KNOBS[k], v) for k, v in PROTOCOL_VALUES.items()
        if FIXED_KNOBS[k] != v}
assert not _bad, ('the harvested engine DISAGREES with GENERALIZATION_PROTOCOL.md '
                  f'on {_bad} -- the generalization test is VOID until resolved')
print('ASSERT OK: every frozen constant in the harvested engine matches '
      'GENERALIZATION_PROTOCOL.md exactly.')


def assert_knobs_unchanged(where):
    '''Re-read every frozen knob out of the live globals and compare.'''
    g = globals()
    now = dict(FIXED_KNOBS)
    for k in ('CONF_L', 'CONFIRM_BARS', 'BAR_DIR_WEIGHT', 'ENSEMBLE_K', 'Z_HI',
              'EFF_HI', 'Z_HI_EXIT', 'EFF_HI_EXIT', 'EFF_WIN', 'CONTEXT_DAYS',
              'INTENSITY_MODE', 'ESCALATION_DURING_HOLD', 'DIRECTION_MODE',
              'TRAIN_FRACTION', 'N_FOLDS', 'H_TARGET_RATE', 'H_EXIT_SLACK',
              'BASE_SEED'):
        now[k] = g[k]
    now['FEATURE_COLS'] = tuple(g['FEATURE_COLS'])
    now['N_FEATURES'] = len(g['FEATURE_COLS'])
    now['DIRECTION_EXCLUDE'] = tuple(g['DIRECTION_EXCLUDE'])
    diff = {k: (FIXED_KNOBS[k], now[k]) for k in FIXED_KNOBS if FIXED_KNOBS[k] != now[k]}
    assert not diff, (f'FIXED KNOB DRIFTED at {where}: {diff}. A knob was '
                      're-tuned per market -- the generalization test is VOID.')


assert_knobs_unchanged('declaration')
print(f'fixed-knob invariant armed over {len(FIXED_KNOBS)} constants; '
      'checked after every run.')

In [ ]:
# ===========================================================================
# BARS_PER_DAY: window rebuilding. The ONLY thing that legitimately changes
# per run, and it changes by the master's OWN formula with the SAME day counts.
# ===========================================================================
BASE_WIN_MASTER = dict(BASE_WIN)     # the master's 2h Nifty windows, for reference


def windows_for(bpd):
    '''The master's BASE_WIN formula, evaluated at a different bars-per-day.'''
    return dict(MOM_1D=MOM_DAYS[0] * bpd, MOM_3D=MOM_DAYS[1] * bpd,
                MOM_5D=MOM_DAYS[2] * bpd,
                VOL_WIN=BASE_WIN_MASTER['VOL_WIN'],      # raw bar counts in the
                VOL_FAST=BASE_WIN_MASTER['VOL_FAST'],    # master; left alone
                VOL_SLOW=CONTEXT_DAYS * bpd,
                SWING_WIN=CONTEXT_DAYS * bpd)


assert windows_for(3) == BASE_WIN_MASTER, \
    'windows_for(3) must reproduce the master BASE_WIN bit for bit'
print('ASSERT OK: windows_for(BARS_PER_DAY=3) reproduces the master BASE_WIN '
      f'exactly -> {BASE_WIN_MASTER}')


@contextlib.contextmanager
def bars_per_day(bpd):
    '''Install this run's day-denominated windows. Restored on exit.'''
    g = globals()
    old_win, old_bpd = g['BASE_WIN'], g['BARS_PER_DAY']
    g['BASE_WIN'], g['BARS_PER_DAY'] = windows_for(bpd), bpd
    try:
        yield g['BASE_WIN']
    finally:
        g['BASE_WIN'], g['BARS_PER_DAY'] = old_win, old_bpd


def fwd_horizons_for(bpd):
    return [max(1, d * bpd) for d in FWD_DAYS]


assert fwd_horizons_for(3) == FWD_HORIZONS, \
    'fwd_horizons_for(3) must reproduce the master FWD_HORIZONS'
print(f'ASSERT OK: fwd_horizons_for(3) == master FWD_HORIZONS == {FWD_HORIZONS}')
print()
print('INTRADAY FX windows that will actually be used:')
for _tf, _b in (('1h', 24), ('2h', 12)):
    _w = windows_for(_b)
    print(f"  {_tf}  BARS_PER_DAY={_b:>3}  momentum {_w['MOM_1D']}/{_w['MOM_3D']}/"
          f"{_w['MOM_5D']} bars   context {_w['SWING_WIN']} bars   "
          f'fwd horizons {fwd_horizons_for(_b)} bars')
print()
print(f'EFF_WIN stays FROZEN at {EFF_WIN} BARS on every run (protocol decree), '
      'i.e. 9h at')
print('1h and 18h at 2h. The chop filter\'s REAL-TIME horizon therefore VARIES '
      'across')
print('the timeframe axis. Same for VOL_WIN=%d and VOL_FAST=%d. Flagged, not fixed.'
      % (BASE_WIN_MASTER['VOL_WIN'], BASE_WIN_MASTER['VOL_FAST']))

In [ ]:
# ===========================================================================
# THE LEAKAGE TRIPWIRE. `label_bars` is SHADOWED by a guard; every later call
# in this notebook goes through it. Four independent runtime checks.
# ===========================================================================
_label_bars_raw = label_bars
LABEL_CALLS = 0
LABEL_PHASE_OPEN = True


@contextlib.contextmanager
def reopen_label_phase(why):
    '''EXPLICITLY and LOUDLY reopen the label phase after the ZigZag exists.

    Used only by the truncation probes, whose whole job is to re-label a cut
    series. Only the PHASE-ORDERING check is relaxed; identity, aliasing and
    the bar_dir_weight decree stay armed, and the reopening prints itself so it
    can never be a silent hole in the proof.
    '''
    global LABEL_PHASE_OPEN
    print('!' * 78)
    print(f'!! LABEL PHASE EXPLICITLY REOPENED: {why}')
    print('!! ordering check relaxed; identity / aliasing / w-decree STAY ARMED')
    print('!' * 78)
    LABEL_PHASE_OPEN = True
    try:
        yield
    finally:
        LABEL_PHASE_OPEN = False
        print('!! LABEL PHASE CLOSED AGAIN -- ordering check re-armed.')


def label_bars(*args, **kwargs):
    '''Guarded shadow of the engine's label_bars. Adds asserts, changes nothing.'''
    global LABEL_CALLS
    assert ZZ_CALLS == 0 or LABEL_PHASE_OPEN, (
        'LEAKAGE: a label was produced AFTER a ZigZag had been computed.')
    _bw = kwargs.get('bar_dir_weight', BAR_DIR_WEIGHT)
    assert _bw == BAR_DIR_WEIGHT, \
        f'BAR_DIR_WEIGHT drifted to {_bw} inside a labelling call'
    for v in list(args) + list(kwargs.values()):
        assert id(v) not in ZZ_IDS, 'LEAKAGE: a ZigZag output was passed to label_bars'
        if isinstance(v, np.ndarray):
            for z in ZZ_ARRAYS:
                assert not np.shares_memory(v, z), \
                    'LEAKAGE: a label_bars argument aliases ZigZag memory'
    LABEL_CALLS += 1
    return _label_bars_raw(*args, **kwargs)


print('leakage tripwire ready.')
print('  check 1  phase ordering  : label_bars asserts ZZ_CALLS == 0')
print('  check 2  object identity : every argument checked against ZZ_IDS')
print('  check 3  memory aliasing : np.shares_memory vs every ZigZag array')
print(f'  check 4  decree          : bar_dir_weight == BAR_DIR_WEIGHT == {BAR_DIR_WEIGHT}')

In [ ]:
# ===========================================================================
# INTRADAY FX LOADER -- REAL DATA, COMMUNITY GITHUB SOURCE.
# NO SYNTHETIC FALLBACK. If the fetch fails, the runs appear as errors.
# ===========================================================================
FX_BASE = ('https://raw.githubusercontent.com/ejtraderLabs/historical-data/'
           'main/{sym}/{sym}{tf}.csv')
FX_SOURCE_LABEL = ('COMMUNITY GITHUB DATA (ejtraderLabs/historical-data) '
                   '- verified against Brexit/COVID, NOT an official feed')

# Plausible MEDIAN band per pair, used to DETECT the integer scaling. These are
# wide, coarse, textbook ranges -- they are a scale detector, not a fit.
FX_PAIRS = [
    dict(sym='EURUSD', name='EUR/USD', med_band=(0.9, 1.7)),
    dict(sym='GBPUSD', name='GBP/USD', med_band=(0.9, 1.7)),
    dict(sym='USDJPY', name='USD/JPY', med_band=(75.0, 160.0)),
]
FX_TFS = [('1h', 1, 24), ('2h', 2, 12)]     # (name, hours, assumed BARS_PER_DAY)

SCALE_CANDIDATES = [1.0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
OHLC_AGG = {'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
            'tick_volume': 'sum'}


def detect_divisor(close_raw, med_band, sym):
    '''Pick the power-of-ten divisor that puts the MEDIAN close in `med_band`.'''
    med = float(np.median(close_raw))
    hits = [d for d in SCALE_CANDIDATES if med_band[0] <= med / d <= med_band[1]]
    assert len(hits) == 1, (
        f'{sym}: scale detection is AMBIGUOUS -- median {med} lands in '
        f'{med_band} for divisors {hits}. Refusing to guess.')
    return hits[0]


def load_fx_h1(p):
    '''Fetch one pair's hourly OHLC, detect + apply the scale, verify.'''
    url = FX_BASE.format(sym=p['sym'], tf='h1')
    df = pd.read_csv(url, parse_dates=['Date'])
    assert list(df.columns) == ['Date', 'open', 'high', 'low', 'close',
                                'tick_volume'], \
        f"{p['sym']}: unexpected columns {list(df.columns)}"
    df = df.set_index('Date').sort_index()
    div = detect_divisor(df['close'].values, p['med_band'], p['sym'])
    px = df[['open', 'high', 'low', 'close']] / div
    # generous sanity band -- the DETECTOR used the median; this checks the
    # WHOLE series did not land somewhere absurd.
    lo, hi = p['med_band'][0] / 1.5, p['med_band'][1] * 1.5
    assert lo <= px.values.min() and px.values.max() <= hi, (
        f"{p['sym']}: rescaled series [{px.values.min():.4f}, "
        f"{px.values.max():.4f}] escapes the sanity band [{lo:.4f}, {hi:.4f}] "
        f'at divisor {div:g}')
    out = px.copy()
    out['tick_volume'] = df['tick_volume'].values
    return out, div


FX_OK, FX_FAIL = True, ''
FX_H1, FX_DIV = {}, {}
try:
    _t = time.time()
    for _p in FX_PAIRS:
        FX_H1[_p['sym']], FX_DIV[_p['sym']] = load_fx_h1(_p)
    print(f'fetched 3 hourly FX files in {time.time() - _t:.1f}s')
except Exception as e:
    FX_OK, FX_FAIL = False, f'{type(e).__name__}: {e}'
    print(f'FX FETCH FAILED: {FX_FAIL}')
    print('The 6 runs will appear in the tables with this reason.')
    print('NOTHING IS SUBSTITUTED. There is no synthetic fallback in this notebook.')

print()
print('=' * 108)
print('PROVENANCE:  ' + FX_SOURCE_LABEL)
print('=' * 108)
print('*** THE DATA ENDS 2022-03-04. This notebook says NOTHING about 2022-2026. ***')
print()
if FX_OK:
    print(f"{'pair':<9} {'divisor':>9} {'bars':>7}  {'span':<44} "
          f"{'median':>9} {'min':>9} {'max':>9}")
    for p in FX_PAIRS:
        d = FX_H1[p['sym']]
        print(f"{p['name']:<9} {FX_DIV[p['sym']]:>9.0f} {len(d):>7}  "
              f"{d.index[0]:%Y-%m-%d %H:%M} -> {d.index[-1]:%Y-%m-%d %H:%M}"
              f"{'':<8}"
              f"{d['close'].median():>9.4f} {d['close'].min():>9.4f} "
              f"{d['close'].max():>9.4f}")
    print()
    print('NOTE: dividing every price by a constant leaves LOG RETURNS unchanged,')
    print('so the divisor cannot alter one feature, one label or one statistic.')
    print('It exists so the charts are readable and so the sanity assert can run.')

In [ ]:
# ===========================================================================
# LIVE DATA VERIFICATION. Structural checks + known-event checks.
# Nothing is trusted on the strength of somebody having checked it before.
# ===========================================================================
FX_CHECKS = []
if FX_OK:
    def _chk(name, ok, detail):
        FX_CHECKS.append((name, bool(ok), detail))
        print(f"  [{'PASS' if ok else 'FAIL'}] {name:<52} {detail}")

    print('STRUCTURAL CHECKS')
    for p in FX_PAIRS:
        d = FX_H1[p['sym']]
        _chk(f"{p['name']} 57,600 hourly bars", len(d) == 57600, f'{len(d)} bars')
        _chk(f"{p['name']} index monotone, no duplicate timestamps",
             d.index.is_monotonic_increasing and not d.index.duplicated().any(),
             f'{d.index[0]:%Y-%m-%d} -> {d.index[-1]:%Y-%m-%d}')
        _chk(f"{p['name']} all prices strictly positive and finite",
             bool((d[['open', 'high', 'low', 'close']] > 0).values.all())
             and bool(np.isfinite(d[['open', 'high', 'low', 'close']].values).all()),
             f"min {d[['open', 'high', 'low', 'close']].values.min():.4f}")
        _hi_ok = (d['high'].values >= np.maximum(d['open'].values, d['close'].values) - 1e-12).all()
        _lo_ok = (d['low'].values <= np.minimum(d['open'].values, d['close'].values) + 1e-12).all()
        _chk(f"{p['name']} OHLC internally consistent",
             bool(_hi_ok and _lo_ok and (d['high'] >= d['low']).all()),
             'high >= max(o,c), low <= min(o,c), high >= low')

    print('\nHOURLY REALISED VOLATILITY (sd of hourly log returns)')
    for p in FX_PAIRS:
        _r = np.diff(np.log(FX_H1[p['sym']]['close'].values))
        print(f"  {p['name']:<9} {_r.std() * 100:.3f}%")

    print('\nEVENT CHECKS against independently known magnitudes')
    _g = FX_H1['GBPUSD']['close'].loc['2016-06-23':'2016-06-24']
    _mv = (_g.min() / _g.max() - 1.0) * 100.0
    _chk('BREXIT GBPUSD 2016-06-23/24 crash',
         abs(_mv - (-10.84)) < 0.30 and abs(_g.max() - 1.5006) < 0.01
         and abs(_g.min() - 1.3379) < 0.01,
         f'{_g.max():.4f} -> {_g.min():.4f} = {_mv:.2f}%  (expected '
         f'1.5006 -> 1.3379 = -10.84%)')

    _e = FX_H1['EURUSD']['close'].loc['2020-03']
    _sw = (_e.max() / _e.min() - 1.0) * 100.0
    _chk('COVID EURUSD March-2020 range',
         abs(_e.min() - 1.0654) < 0.01 and abs(_e.max() - 1.1470) < 0.01
         and abs(_sw - 7.7) < 0.5,
         f'{_e.min():.4f} - {_e.max():.4f} ({_sw:.1f}% swing)  '
         f'(expected 1.0654 - 1.1470, 7.7%)')

    _np_ = sum(1 for _, o, _ in FX_CHECKS if o)
    print(f'\nDATA VERIFICATION: {_np_}/{len(FX_CHECKS)} checks PASS')
    assert _np_ == len(FX_CHECKS), (
        'a data verification check FAILED -- the source is not what it was '
        'verified to be. STOP: do not proceed on data you cannot confirm.')
    print('ASSERT OK: every structural and event check passed on data fetched '
          'in THIS run.')
    print()
    print('This is a COMMUNITY repository. What these checks establish is that '
          'its')
    print('content reproduces two large, independently-known FX moves to within '
          'a few')
    print('basis points, and that it is structurally clean. They do NOT make it '
          'an')
    print('official feed, and it is not labelled as one anywhere in this notebook.')
else:
    print('data unavailable -- verification skipped, runs will be error rows.')

In [ ]:
# ===========================================================================
# DOWNWARD-ONLY RESAMPLING + the causal realised-vol VIX proxy.
# ===========================================================================
def resample_ohlc_down(df, hours):
    '''Downward OHLC aggregation from hourly bars. NEVER upward.

    open=first, high=max, low=min, close=last, tick_volume=sum.
    Asserts (a) the output cadence is COARSER than the input, and (b) the bar
    count falls by roughly the expected factor. Finer bars are never
    manufactured from coarser ones.
    '''
    assert hours >= 2, 'resample_ohlc_down is for COARSENING only'
    step_in = pd.Series(df.index).diff().median()
    out = df.resample(f'{hours}h').agg(OHLC_AGG).dropna(how='any')
    step_out = pd.Series(out.index).diff().median()
    assert step_out > step_in, (f'UPWARD resampling attempted: {step_in} -> '
                                f'{step_out}. Only downward is allowed.')
    ratio = len(out) / len(df)
    assert 0.35 <= ratio * hours <= 1.15, (
        f'{hours}h resample produced {len(out)} bars from {len(df)} '
        f'(x{ratio:.3f}); expected roughly x{1 / hours:.3f}')
    assert (out['high'].values >= np.maximum(out['open'].values, out['close'].values) - 1e-12).all()
    assert (out['low'].values <= np.minimum(out['open'].values, out['close'].values) + 1e-12).all()
    return out, ratio


def realised_vol_proxy(close, bpd, win=None):
    '''CAUSAL trailing realised-volatility stand-in for a VIX.

    ann.vol(t) = sd(log returns over the last `win` bars, inclusive of t)
                 * sqrt(bpd * 252) * 100

    Reads bars <= t only: the return at bar t is log(C[t]/C[t-1]) and the
    rolling window ends at t. There is no shift, no centring and no bfill.
    This is NOT a VIX. It is labelled `vol-proxy` everywhere it is used.
    '''
    win = int(BASE_WIN['VOL_SLOW'] if win is None else win)
    r = np.log(close / close.shift(1))
    v = r.rolling(win).std() * np.sqrt(max(bpd, 1) * 252.0) * 100.0
    v = v.replace([np.inf, -np.inf], np.nan)
    # leading NaNs are left as NaN -- build_features' dropna() removes exactly
    # those bars. NEVER bfill: that is the look-ahead the master already fixed.
    return v.rename('vix')


def realised_bpd(index):
    '''Median number of bars per calendar SESSION actually present in the data.'''
    if len(index) == 0:
        return np.nan
    per_day = pd.Series(1, index=index).groupby(index.normalize()).sum()
    return float(per_day.median())


if FX_OK:
    print('DOWNWARD RESAMPLE CHECK (1h source -> 2h)')
    print(f"{'pair':<9} {'1h bars':>8} {'2h bars':>8} {'ratio':>7}  cadence")
    FX_H2 = {}
    for p in FX_PAIRS:
        d2, rr = resample_ohlc_down(FX_H1[p['sym']], 2)
        FX_H2[p['sym']] = d2
        _s1 = pd.Series(FX_H1[p['sym']].index).diff().median()
        _s2 = pd.Series(d2.index).diff().median()
        print(f"{p['name']:<9} {len(FX_H1[p['sym']]):>8} {len(d2):>8} "
              f"{rr:>7.3f}  {_s1} -> {_s2}")
    print('\nASSERT OK: every 2h series is a DOWNWARD aggregation of its own 1h')
    print('file, the cadence strictly coarsens, and the bar count roughly halves.')
    print('No finer bar is fabricated from a coarser one anywhere in this notebook.')

print()
print('VOLATILITY INPUT for every run: realised-vol PROXY (causal).')
print('Spot FX has NO natural VIX. The vix_chg feature is fed a trailing')
print('realised-volatility proxy that reads bars <= t only. It is NOT a VIX and')
print('is stamped as a proxy on every run and in every table.')

In [ ]:
# ===========================================================================
# THE RUN HARNESS. Fits + labels ONE (pair, timeframe) run.
# ===========================================================================
N_STATES_FIXED = FIXED_KNOBS['N_STATES']
COV_FIXED = FIXED_KNOBS['COVARIANCE']

FIT_COUNT = 0

# TRAIN_CAP_BARS: None = no cap (fit on the full leading TRAIN_FRACTION window).
# Set only by the runtime probe below, and only if the projection is excessive.
TRAIN_CAP_BARS = None
RUNTIME_BUDGET_S = 15 * 60


def seed_sets(K=ENSEMBLE_K, R=R_SEED_SETS, base=BASE_SEED):
    ss = [list(range(base + r * K, base + r * K + K)) for r in range(R)]
    flat = [s for st in ss for s in st]
    assert len(set(flat)) == len(flat), 'seed sets must be DISJOINT'
    return ss


SEED_SETS = seed_sets()
print(f'R={R_SEED_SETS} DISJOINT seed sets, K={ENSEMBLE_K}: {SEED_SETS}')


def prepare_run(close, volser, bpd):
    '''Features + scaler + fit window for one run. Causal throughout.'''
    with bars_per_day(bpd):
        feat = build_features(close, volser, LOOKBACK_SCALE)
    if len(feat) < 200:
        raise ValueError(f'only {len(feat)} feature bars after warmup (need >= 200)')
    dates = feat.index
    n_bars = len(dates)
    n_fit = max(int(n_bars * TRAIN_FRACTION), 50)
    raw = feat[FEATURE_COLS].values
    # The fit window is the LEADING n_fit bars. A cap, if armed, shortens it
    # from the LEFT (most recent bars kept), and is applied FORWARD identically
    # on every run. n_fit itself -- the IS/OOS boundary -- never moves.
    fit_lo = 0 if TRAIN_CAP_BARS is None else max(0, n_fit - int(TRAIN_CAP_BARS))
    sc = StandardScaler().fit(raw[fit_lo:n_fit])
    Xs = sc.transform(raw)
    return dict(feat=feat, dates=dates, n_bars=n_bars, n_fit=n_fit, fit_lo=fit_lo,
                Xs=Xs, scaler=sc,
                close=close.reindex(dates), trend_raw=feat[TREND_FEATURE].values)


def label_run(prep, close, bpd, models_by_set=None):
    '''Label the full series once per seed set. Returns (label_sets, models).'''
    global FIT_COUNT
    out, mods = [], []
    for si, sd in enumerate(SEED_SETS):
        if models_by_set is None:
            ms, _conv = fit_hmm_ensemble(prep['Xs'][prep['fit_lo']:prep['n_fit']],
                                         N_STATES_FIXED, COV_FIXED,
                                         K=ENSEMBLE_K, base_seed=sd[0])
            FIT_COUNT += ENSEMBLE_K
        else:
            ms = models_by_set[si]
        mods.append(ms)
        with bars_per_day(bpd):
            lab = label_bars(ms, prep['Xs'], prep['dates'], close,
                             prep['trend_raw'], prep['n_fit'], FEATURE_COLS,
                             dir_feats=prep['feat'])
        out.append(lab['tactical_regime_state'].values.copy())
    return out, mods


def build_run(run_id, pair, tf, close, volser, vix_kind, bpd_assumed,
              source, note=''):
    '''Full LABEL-PHASE work for one run. Returns a run dict (or an error row).'''
    rec = dict(run_id=run_id, market=pair, tf=tf, vix_kind=vix_kind,
               source=source, note=note, bpd_assumed=bpd_assumed,
               bpd_realised=realised_bpd(close.index), error=None)
    rec['_raw_close'] = close
    rec['_raw_vol'] = volser
    try:
        prep = prepare_run(close, volser, bpd_assumed)
        labels, models = label_run(prep, close, bpd_assumed)
        rec.update(labels=labels, models=models,
                   dates=prep['dates'], close_arr=prep['close'].values,
                   close_ser=prep['close'], feat=prep['feat'],
                   Xs=prep['Xs'], scaler=prep['scaler'],
                   n_bars=prep['n_bars'], n_fit=prep['n_fit'],
                   fit_lo=prep['fit_lo'],
                   span=(prep['dates'][0], prep['dates'][-1]),
                   win=windows_for(bpd_assumed),
                   fwd_h=fwd_horizons_for(bpd_assumed))
    except Exception as e:
        rec['error'] = f'{type(e).__name__}: {e}'
    assert_knobs_unchanged(f'after run {run_id}')
    return rec


print('run harness ready: prepare_run / label_run / build_run.')
print('Per-run inputs are ONLY (close, volatility, BARS_PER_DAY). No knob is a '
      'function of the pair.')

In [ ]:
# ===========================================================================
# EVAL PHASE BEGINS.
# ===========================================================================
LABEL_PHASE_OPEN = False
print('LABEL PHASE CLOSED. Any label_bars call from here on fails the ordering '
      'check unless the phase is EXPLICITLY reopened.')

BULL_SET = ('H_BULL', 'L_BULL')
BEAR_SET = ('H_BEAR', 'L_BEAR')


def fwd_returns(close_arr, h):
    f = np.full(len(close_arr), np.nan)
    if h < len(close_arr):
        f[:len(close_arr) - h] = (close_arr[h:] / close_arr[:len(close_arr) - h]
                                  - 1.0) * 100.0
    return f


def span_contrast(labels, close_arr, horizons, lo, hi):
    '''HAC contrast over bars [lo, hi). Returns dict per h + the h-mean.

    The forward return is computed on the FULL close array first and then
    sliced, so a bar near the right edge of the span uses the real next price
    rather than a truncated one. The EMBARGO applied by the caller is what
    stops an IS bar from reading an OOS price.
    '''
    labels = np.asarray(labels, dtype=object)
    grp = np.where(np.isin(labels, BULL_SET), 1.0,
                   np.where(np.isin(labels, BEAR_SET), -1.0, 0.0))
    per_h, ts, ds = {}, [], []
    for h in horizons:
        f = fwd_returns(close_arr, h)
        d, se, t, npos, nneg = _w_hac_contrast(f[lo:hi], grp[lo:hi], lag=h - 1)
        per_h[h] = dict(diff_pct=d, hac_se=se, hac_t=t, n_bull=npos, n_bear=nneg)
        if np.isfinite(t):
            ts.append(t)
        if np.isfinite(d):
            ds.append(d)
    return dict(per_h=per_h,
                t_mean=float(np.mean(ts)) if ts else np.nan,
                d_mean=float(np.mean(ds)) if ds else np.nan,
                n=int(hi - lo))


ORDER = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']


def ordering_status(labels, close_arr, horizons, lo, hi):
    '''5-label monotonicity of mean forward return, averaged over horizons.'''
    labels = np.asarray(labels, dtype=object)[lo:hi]
    means = {}
    for lb in ORDER:
        m = (labels == lb)
        if m.sum() < 5:
            means[lb] = np.nan
            continue
        vs = []
        for h in horizons:
            f = fwd_returns(close_arr, h)[lo:hi]
            v = f[m]
            v = v[np.isfinite(v)]
            if len(v):
                vs.append(float(np.mean(v)))
        means[lb] = float(np.mean(vs)) if vs else np.nan
    seq = [means[lb] for lb in ORDER]
    ok = [s for s in seq if np.isfinite(s)]
    if len(ok) < 4:
        return 'THIN', means
    fully = all(a >= b - 1e-12 for a, b in zip(ok, ok[1:]))
    bull = [means[l] for l in BULL_SET if np.isfinite(means[l])]
    bear = [means[l] for l in BEAR_SET if np.isfinite(means[l])]
    block = bool(bull and bear and min(bull) >= max(bear))
    return ('HOLDS' if fully else ('PARTIAL' if block else 'BROKEN')), means


def evaluate_run(r):
    '''S / L / W, occupancy, guards, and the IS-vs-OOS overfitting test.'''
    if r['error']:
        return r
    labels, close_arr = r['labels'], r['close_arr']
    horizons = r['fwd_h']
    n, n_fit = r['n_bars'], r['n_fit']
    embargo = max(horizons)
    is_lo, is_hi = 0, max(50, n_fit - embargo)
    oos_lo, oos_hi = n_fit, n

    # ---- S / L / W / occupancy / guards -- verbatim metrics ----------------
    S, Smat, pair_vals = metric_S(labels)
    swings = zigzag_swings(close_arr, ZZ_PCT)
    Lm = [metric_L(lb, swings)[0] for lb in labels]
    L75 = [metric_L(lb, swings)[1] for lb in labels]
    unm = [metric_L(lb, swings)[3] for lb in labels]
    Ws = [metric_W(lb) for lb in labels]
    occs = [occupancy_pct(lb) for lb in labels]
    occ = {lb: float(np.mean([o[lb] for o in occs])) for lb in REGIME_LABELS}
    W = float(np.mean(Ws))
    g = evaluate_guards(occ, W)

    # ---- the overfitting test, PER SEED SET, IS and OOS SEPARATELY ---------
    IS, OOS = [], []
    for lb in labels:
        IS.append(span_contrast(lb, close_arr, horizons, is_lo, is_hi))
        OOS.append(span_contrast(lb, close_arr, horizons, oos_lo, oos_hi))
    is_t = [x['t_mean'] for x in IS]
    oos_t = [x['t_mean'] for x in OOS]
    is_d = [x['d_mean'] for x in IS]
    oos_d = [x['d_mean'] for x in OOS]

    ord_is, means_is = ordering_status(labels[0], close_arr, horizons, is_lo, is_hi)
    ord_oos, means_oos = ordering_status(labels[0], close_arr, horizons, oos_lo, oos_hi)

    def _mn(v):
        v = [x for x in v if np.isfinite(x)]
        return float(np.mean(v)) if v else np.nan

    def _sp(v):
        v = [x for x in v if np.isfinite(x)]
        return (float(np.max(v) - np.min(v)) if len(v) > 1 else np.nan)

    IS_T, OOS_T = _mn(is_t), _mn(oos_t)
    IS_D, OOS_D = _mn(is_d), _mn(oos_d)
    inverted = bool(np.isfinite(IS_D) and np.isfinite(OOS_D)
                    and np.sign(IS_D) != np.sign(OOS_D) and IS_D != 0 and OOS_D != 0)

    r.update(S=S, S_matrix=Smat, S_min=min(pair_vals), S_max=max(pair_vals),
             L=float(np.mean(Lm)), L75=float(np.mean(L75)),
             unmatched=float(np.mean(unm)), n_swings=len(swings),
             W=W, W_lo=float(np.min(Ws)), W_hi=float(np.max(Ws)),
             occ=occ, guards=g, guards_ok=guards_pass(g),
             IS_T=IS_T, OOS_T=OOS_T, IS_D=IS_D, OOS_D=OOS_D,
             IS_T_spread=_sp(is_t), OOS_T_spread=_sp(oos_t),
             IS_n=IS[0]['n'], OOS_n=OOS[0]['n'], embargo=embargo,
             ord_is=ord_is, ord_oos=ord_oos,
             means_is=means_is, means_oos=means_oos,
             inverted=inverted, IS_perh=IS[0]['per_h'], OOS_perh=OOS[0]['per_h'])
    return r



## 2. The disjoint feature set — added alongside the shipped one, neither modified in place

In [ ]:
# ===========================================================================
# THE DISJOINT-BLOCK FEATURE SET.
#
# d0_1/d1_5/d5_20/d20_60 are NON-OVERLAPPING return blocks built from nested
# cumulative sums by subtraction:
#   d0_1   = cumsum(0->1d)
#   d1_5   = cumsum(0->5d)  - cumsum(0->1d)
#   d5_20  = cumsum(0->20d) - cumsum(0->5d)
#   d20_60 = cumsum(0->60d) - cumsum(0->20d)
# Measured (protocol): max |off-diagonal corr| 0.588 -> 0.036, condition
# number 6.1 -> 1.1, effective dims (participation ratio) 2.74 -> 4.00 of 4,
# at the SAME 4 horizons -- disjointness, not the horizons, buys the rank.
#
# vol_2h, vix_chg, dist_ma are UNCHANGED formulas (same windows). drawdown is
# DROPPED (-0.86 corr with dist_ma -- keep one, not both). ret_2h is dropped
# (subsumed by d0_1, which spans the same first-day window).
#
# TREND_FEATURE ('mom_3d') is still produced, at ITS ORIGINAL 3-day window,
# for the INTENSITY (H vs L) gate ONLY -- deliberately NOT one of the 7 HMM
# inputs. That axis already works (E1 in the diagnostic probes: W falls
# monotonically as eff_fast rises) and is left untouched. One mechanism at a
# time.
# ===========================================================================
DISJOINT_EDGE_DAYS = dict(E1=1, E5=5, E20=20, E60=60)


def build_features_disjoint(close, vix, scale=LOOKBACK_SCALE):
    '''7 causal features: near-orthogonal disjoint return blocks + vol_2h +
    vix_chg + dist_ma. mom_3d is also produced, for TREND_FEATURE only.'''
    bpd, cd = BARS_PER_DAY, CONTEXT_DAYS
    e = {k: max(2, int(round(d * bpd * scale)))
         for k, d in DISJOINT_EDGE_DAYS.items()}
    swing = max(2, int(round(cd * bpd * scale)))
    mom3_w = max(2, int(round(3 * bpd * scale)))
    vol_w = max(2, int(round(10 * scale)))
    r = np.log(close / close.shift(1))
    cum1, cum5 = r.rolling(e['E1']).sum(), r.rolling(e['E5']).sum()
    cum20, cum60 = r.rolling(e['E20']).sum(), r.rolling(e['E60']).sum()
    df = pd.DataFrame(index=close.index)
    df['d0_1'] = cum1
    df['d1_5'] = cum5 - cum1
    df['d5_20'] = cum20 - cum5
    df['d20_60'] = cum60 - cum20
    df['vol_2h'] = r.rolling(vol_w).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    ma = close.rolling(swing).mean()
    df['dist_ma'] = (close - ma) / ma
    df['mom_3d'] = r.rolling(mom3_w).sum()      # TREND_FEATURE only
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


DISJOINT_FEATURE_COLS = ['d0_1', 'd1_5', 'd5_20', 'd20_60', 'vol_2h',
                         'vix_chg', 'dist_ma']
DISJOINT_FEATURE_SIGN = {'d0_1': 1.0, 'd1_5': 1.0, 'd5_20': 1.0,
                         'd20_60': 1.0, 'vol_2h': -1.0, 'vix_chg': -1.0,
                         'dist_ma': 1.0}
DISJOINT_FEATURE_MAG = {'d0_1': 0.4, 'd1_5': 0.4, 'd5_20': 0.4, 'd20_60': 0.4}
DISJOINT_DIRECTION_EXCLUDE = ('vol_2h',)
DISJOINT_BAR_DIR_FEATURES = ('d0_1', 'd1_5', 'd5_20', 'd20_60', 'dist_ma')

# n_params arithmetic -- F4, trivially confirms, printed for the record.
_p_nested = n_params(5, len(FEATURE_COLS), 'diag')
_p_disjoint = n_params(5, len(DISJOINT_FEATURE_COLS), 'diag')
print(f'F4  n_params(N=5, diag): nested {len(FEATURE_COLS)} feat -> '
      f'{_p_nested:.0f} params   disjoint {len(DISJOINT_FEATURE_COLS)} feat '
      f'-> {_p_disjoint:.0f} params')
assert _p_disjoint < _p_nested
print(f'F4  CONFIRMED: {_p_disjoint:.0f} < {_p_nested:.0f}')

DESIGNS = {
    'nested': dict(FEATURE_COLS=list(FEATURE_COLS), FEATURE_SIGN=dict(FEATURE_SIGN),
                   FEATURE_MAG=dict(FEATURE_MAG),
                   DIRECTION_EXCLUDE=tuple(DIRECTION_EXCLUDE),
                   BAR_DIR_FEATURES=tuple(BAR_DIR_FEATURES),
                   build_features=build_features),
    'disjoint': dict(FEATURE_COLS=list(DISJOINT_FEATURE_COLS),
                     FEATURE_SIGN=dict(DISJOINT_FEATURE_SIGN),
                     FEATURE_MAG=dict(DISJOINT_FEATURE_MAG),
                     DIRECTION_EXCLUDE=tuple(DISJOINT_DIRECTION_EXCLUDE),
                     BAR_DIR_FEATURES=tuple(DISJOINT_BAR_DIR_FEATURES),
                     build_features=build_features_disjoint),
}
SWEPT_KEYS = tuple(DESIGNS['nested'].keys())


@contextlib.contextmanager
def feature_design(name):
    '''Install one feature-set arm. Re-arms the fixed-knob invariant around
    exactly the keys the design touches; every other knob stays guarded.'''
    g = globals()
    spec = DESIGNS[name]
    old = {k: g[k] for k in SWEPT_KEYS}
    old_fixed = dict(FIXED_KNOBS)
    g.update(spec)
    FIXED_KNOBS['FEATURE_COLS'] = tuple(spec['FEATURE_COLS'])
    FIXED_KNOBS['N_FEATURES'] = len(spec['FEATURE_COLS'])
    FIXED_KNOBS['DIRECTION_EXCLUDE'] = tuple(spec['DIRECTION_EXCLUDE'])
    try:
        yield name
    finally:
        g.update(old)
        FIXED_KNOBS.clear()
        FIXED_KNOBS.update(old_fixed)


print(f"\nDESIGNS installed: {list(DESIGNS)} -- swept keys: {SWEPT_KEYS}")
print(f"nested   FEATURE_COLS ({len(DESIGNS['nested']['FEATURE_COLS'])}): "
      f"{DESIGNS['nested']['FEATURE_COLS']}")
print(f"disjoint FEATURE_COLS ({len(DESIGNS['disjoint']['FEATURE_COLS'])}): "
      f"{DESIGNS['disjoint']['FEATURE_COLS']}")

## 3. Bit-for-bit equivalence: `feature_design('nested')` must be a true no-op

In [ ]:
# ===========================================================================
# Every new switch needs an OFF value asserted bit-for-bit against the
# unmodified engine. Run EUR/USD@1h WITHOUT the context manager (baseline,
# harvested engine exactly as build_intraday_fx.py left it), then again
# INSIDE feature_design('nested'), and diff every label.
# ===========================================================================
assert_knobs_unchanged('before feature_design exists')
FEATURE_COLS_BASELINE = list(FEATURE_COLS)

_probe_close = FX_H1['EURUSD']['close'].astype(float)
_probe_close.name = 'EURUSD'
with bars_per_day(24):
    _probe_vol = realised_vol_proxy(_probe_close, 24)

_base_prep = prepare_run(_probe_close, _probe_vol, 24)
_base_labels, _ = label_run(_base_prep, _probe_close, 24)

with feature_design('nested'):
    assert_knobs_unchanged("inside feature_design('nested')")
    _nest_prep = prepare_run(_probe_close, _probe_vol, 24)
    _nest_labels, _ = label_run(_nest_prep, _probe_close, 24)
assert_knobs_unchanged("after feature_design('nested')")

_diff = sum(int((a != b).sum()) for a, b in zip(_base_labels, _nest_labels))
_total = sum(len(a) for a in _base_labels)
print(f"EUR/USD@1h baseline vs feature_design('nested'): {_diff} of {_total} "
      f"labels differ across {len(_base_labels)} seed sets")
assert _diff == 0, "feature_design('nested') is NOT a bit-for-bit no-op"
print("ASSERT OK: feature_design('nested') reproduces the unmodified engine "
      "exactly.")

with feature_design('disjoint'):
    assert CONTEXT_DAYS == 12
    assert windows_for(24)['SWING_WIN'] == 12 * 24, \
        'CONTEXT_DAYS did not reach the disjoint dist_ma window'
    _dj_test = prepare_run(_probe_close, _probe_vol, 24)
    assert 'd0_1' in _dj_test['feat'].columns and 'mom_3d' in _dj_test['feat'].columns
    assert list(_dj_test['feat'][FEATURE_COLS].columns) == DISJOINT_FEATURE_COLS
print("ASSERT OK: feature_design('disjoint') installs, produces d0_1..d20_60 "
      "+ mom_3d (TREND_FEATURE only), and restores cleanly.")
assert FEATURE_COLS == list(FEATURE_COLS_BASELINE), \
    'feature_design failed to restore FEATURE_COLS after the disjoint arm'
del _base_prep, _base_labels, _nest_prep, _nest_labels, _dj_test

## 4. The instrument set — all 7, graded together (not a search)

In [ ]:
# ===========================================================================
# ALL 7 non-redundant instruments in the repo (same set fx_replication.ipynb
# assembled: crosses and 2h excluded for their measured pseudo-replication
# reasons). No discovery/held-out split -- see the protocol for why.
# ===========================================================================
ALL_INST = [
    dict(sym='EURUSD', name='EUR/USD', med_band=(0.9, 1.7),     cls='FX'),
    dict(sym='GBPUSD', name='GBP/USD', med_band=(0.9, 1.7),     cls='FX'),
    dict(sym='USDJPY', name='USD/JPY', med_band=(75.0, 160.0),  cls='FX'),
    dict(sym='XAUUSD', name='XAU/USD', med_band=(1000., 2100.), cls='COMMODITY'),
    dict(sym='AUDUSD', name='AUD/USD', med_band=(0.55, 1.15),   cls='FX'),
    dict(sym='USDCAD', name='USD/CAD', med_band=(0.9, 1.6),     cls='FX'),
    dict(sym='USDCHF', name='USD/CHF', med_band=(0.7, 1.15),    cls='FX'),
]
BPD_1H, TF_NAME = 24, '1h'

FS_H1, FS_DIV = {}, {}
_t0 = time.time()
for d in ALL_INST:
    FS_H1[d['sym']], FS_DIV[d['sym']] = load_fx_h1(d)
print(f'fetched {len(ALL_INST)} hourly series in {time.time() - _t0:.1f}s')
print(f"\n{'instrument':<11}{'class':<12}{'divisor':>9}{'bars':>8}  span"
      f"{'':<18}{'median':>10}")
for d in ALL_INST:
    b = FS_H1[d['sym']]
    print(f"{d['name']:<11}{d['cls']:<12}{FS_DIV[d['sym']]:>9g}{len(b):>8}  "
          f"{b.index[0].date()} -> {b.index[-1].date()}"
          f"{b['close'].median():>10.4f}")
for d in ALL_INST:
    b = FS_H1[d['sym']]
    assert b.index.is_monotonic_increasing and not b.index.duplicated().any()
    assert (b[['open', 'high', 'low', 'close']] > 0).all().all()
print('\nASSERT OK: all 7 series monotone, de-duplicated, positive.')
print('*** ALL SERIES END 2022-03-04. Nothing here speaks to 2022-2026. ***')

In [ ]:
# ===========================================================================
# PRE-REGISTERED, frozen in protocols/FX_FEATURESET_PROTOCOL.md.
# ===========================================================================
F_PREDICTIONS = {
 'F1': ('PRIMARY. In the "steady trend, low eff_fast / high eff_slow" cell, '
        'W is LOWER for disjoint than nested on >= 4 of 7 instruments AND on '
        'the pooled mean.',
        'A pooled drop with the majority failing is reported as a split, '
        'not a confirmation.'),
 'F2': ('Disjoint still shows W falling monotonically Q1->Q5 of eff_fast, '
        'pooled across instruments -- the violent-move axis is not broken.',
        'CONFIRMED if the pooled Q1->Q5 mean W is monotone decreasing.'),
 'F3': ('Disjoint has a LOWER feature-correlation condition number and a '
        'HIGHER participation-ratio effective-dimension share than nested, '
        'measured live, on >= 4 of 7 instruments.',
        'Not assumed from the earlier AUD/USD-only probe.'),
 'F4': ('n_params(disjoint) < n_params(nested) at N=5, diag.',
        'Arithmetic; already shown above.'),
 'F5': ('Guards G1-G4 pass on disjoint on AT LEAST as many of the 7 runs as '
        'on nested.',
        'A coherence win that breaks a guard is not reported as a win.'),
 'F6': ('Mean seed-stability S is higher for disjoint than nested, pooled '
        'across the 7 instruments.',
        'The bistability hypothesis PROJECT_STATE has not resolved.'),
 'F7': ('HONESTY CHECK, not the goal. Disjoint OOS (BULL-BEAR) HAC t is not '
        'systematically worse than nested (mean difference >= -0.20).',
        'Not expected to manufacture skill fx_meanrev.ipynb already found '
        'absent. Does not gate F1-F6.'),
}
print('=' * 108)
print('PRE-REGISTERED PREDICTIONS -- printed before a single run exists')
print('=' * 108)
for k, (claim, rule) in F_PREDICTIONS.items():
    print(f'{k}.  {claim}')
    print(f'    RULE: {rule}')

## 5. Runtime measurement — measured first, then extrapolated, then run

In [ ]:
# ===========================================================================
# RUNTIME PROBE. One seed set, NESTED design (9 features, more than disjoint's
# 7 -- an upper bound on per-fit cost), on the largest instrument.
# ===========================================================================
TRAIN_CAP_BARS = None
RUNTIME_EST = None

_p0 = ALL_INST[0]
_c0 = FS_H1[_p0['sym']]['close'].astype(float)
_c0.name = _p0['sym']
with bars_per_day(BPD_1H):
    _v0 = realised_vol_proxy(_c0, BPD_1H)

with feature_design('nested'):
    _t = time.time()
    _prep0 = prepare_run(_c0, _v0, BPD_1H)
    _t_prep = time.time() - _t
    _t = time.time()
    _m0, _ = fit_hmm_ensemble(_prep0['Xs'][_prep0['fit_lo']:_prep0['n_fit']],
                              N_STATES_FIXED, COV_FIXED, K=ENSEMBLE_K,
                              base_seed=SEED_SETS[0][0])
    _t_set = time.time() - _t
    FIT_COUNT += ENSEMBLE_K
    _t = time.time()
    with bars_per_day(BPD_1H):
        _l0 = label_bars(_m0, _prep0['Xs'], _prep0['dates'], _c0,
                         _prep0['trend_raw'], _prep0['n_fit'], FEATURE_COLS,
                         dir_feats=_prep0['feat'])
    _t_lab = time.time() - _t

_per_run = R_SEED_SETS * (_t_set + _t_lab) + _t_prep
N_RUNS_TOTAL = len(DESIGNS) * len(ALL_INST)
RUNTIME_EST = N_RUNS_TOTAL * _per_run

print('=' * 100)
print('RUNTIME PROBE -- measured before anything is launched')
print('=' * 100)
print(f"  probe run              : {_p0['name']} @ 1h, nested design")
print(f"  feature bars           : {_prep0['n_bars']}   training rows: "
      f"{_prep0['n_fit'] - _prep0['fit_lo']}")
print(f'  prepare_run            : {_t_prep:.1f}s')
print(f'  ONE seed set (K={ENSEMBLE_K} fits): {_t_set:.1f}s')
print(f'  label_bars full series : {_t_lab:.1f}s')
print(f'  projected per run      : {_per_run:6.0f}s  (R={R_SEED_SETS} seed sets)')
print(f'  {N_RUNS_TOTAL} runs ({len(DESIGNS)} designs x {len(ALL_INST)} '
      f'instruments) -> projected {RUNTIME_EST:.0f}s (~{RUNTIME_EST / 60:.1f} min)')

RUNTIME_BUDGET_S = 25 * 60
if RUNTIME_EST > RUNTIME_BUDGET_S:
    TRAIN_CAP_BARS = 12000     # declared constant -- NEVER wall-clock-derived
    print(f'\nOVER BUDGET ({RUNTIME_BUDGET_S}s) -- capping the training window '
          f'to the most recent {TRAIN_CAP_BARS} bars. R=4 stays intact; no '
          'instrument or design is dropped.')
else:
    print(f'\nUnder the {RUNTIME_BUDGET_S}s budget -- no cap applied. Every run '
          'uses its full leading training window.')

## 6. Label phase — 2 designs × 7 instruments = 14 runs

In [ ]:
# ===========================================================================
# LABEL PHASE. Every run: (design, instrument) -> prepare_run + label_run
# inside feature_design(design). CONTEXT_DAYS=12 for BOTH designs -- the only
# thing that changes is feature construction.
# ===========================================================================
FEATSET_RUNS = []
_t0 = time.time()
print(f"{'design':<10}{'instrument':<11}{'bars':>8}{'fit rows':>10}{'secs':>8}")
print('-' * 50)
for design in DESIGNS:
    for d in ALL_INST:
        _tt = time.time()
        close = FS_H1[d['sym']]['close'].astype(float)
        close.name = d['sym']
        with feature_design(design):
            with bars_per_day(BPD_1H):
                vol = realised_vol_proxy(close, BPD_1H)
            prep = prepare_run(close, vol, BPD_1H)
            labels, _m = label_run(prep, close, BPD_1H)
            assert_knobs_unchanged(f"{d['name']} {design}")
        FEATSET_RUNS.append(dict(
            run_id=f"{d['name']}|{design}", inst=d['name'], design=design,
            cls=d['cls'], tf=TF_NAME, error=None, labels=labels,
            close_arr=prep['close'].values, dates=prep['dates'],
            feat=prep['feat'][list(DESIGNS[design]['FEATURE_COLS'])].copy(),
            fwd_h=fwd_horizons_for(BPD_1H),
            n_bars=prep['n_bars'], n_fit=prep['n_fit']))
        print(f"{design:<10}{d['name']:<11}{prep['n_bars']:>8}"
              f"{prep['n_fit']:>10}{time.time() - _tt:>8.1f}")
print('-' * 50)
print(f'LABEL PHASE: {time.time() - _t0:.1f}s   {len(FEATSET_RUNS)} runs   '
      f'{FIT_COUNT} HMM fits total (incl. probes)   TRAIN_CAP_BARS = {TRAIN_CAP_BARS}')
assert ZZ_CALLS == 0, 'LEAKAGE: a ZigZag existed during the label phase'
print('ASSERT OK: ZZ_CALLS == 0 -- every label produced before any ZigZag.')

## 7. Eval phase — S/L/W/guards/IS-OOS (verbatim), plus the grind-coherence metrics

In [ ]:
# ===========================================================================
# EVAL PHASE -- strictly after the whole label phase.
# ===========================================================================
LABEL_PHASE_OPEN = False
for r in FEATSET_RUNS:
    evaluate_run(r)
print(f'evaluated {len(FEATSET_RUNS)} runs   ZZ_CALLS={ZZ_CALLS}')

In [ ]:
# ===========================================================================
# GRIND-COHERENCE METRICS (F1/F2) -- same eff_fast/eff_slow construction
# already used (and pre-registered) in the diagnostic probes: eff_fast over
# EFF_WIN=9 bars (what the chop filter sees), eff_slow over 20 DAYS (the
# trend a human sees on the chart). Uses labels[0] and the OOS span only,
# same convention as ord_is/ord_oos/means_is/means_oos already use in
# evaluate_run.
# ===========================================================================
EFF_FAST_WIN = EFF_WIN            # 9 bars, the shipped chop-filter horizon
EFF_SLOW_WIN = 20 * BPD_1H        # 20 days


def efficiency(px, w):
    lp = np.log(px)
    d = np.abs(np.diff(lp, prepend=lp[0]))
    net = np.full(len(px), np.nan)
    net[w:] = np.abs(lp[w:] - lp[:-w])
    tot = np.convolve(d, np.ones(w), 'full')[:len(px)]
    with np.errstate(invalid='ignore', divide='ignore'):
        e = net / tot
    return np.clip(e, 0, 1)


def feature_conditioning(X):
    '''(condition number, participation-ratio effective dims) of a raw
    feature matrix -- correlation eigenvalues, no NaNs.'''
    C = np.corrcoef(X, rowvar=False)
    ev = np.clip(np.linalg.eigvalsh(C), 1e-12, None)
    cond = float(ev.max() / ev.min())
    eff_dims = float(ev.sum() ** 2 / (ev ** 2).sum())
    return cond, eff_dims


MIN_CELL_BARS = 300
for r in FEATSET_RUNS:
    lab0 = np.asarray(r['labels'][0], object)
    px = r['close_arr']
    lo = r['n_fit']
    ef, es = efficiency(px, EFF_FAST_WIN), efficiency(px, EFF_SLOW_WIN)
    sw = np.zeros(len(lab0))
    sw[1:] = (lab0[1:] != lab0[:-1]).astype(float)
    base = np.zeros(len(px), bool)
    base[lo:] = True
    base &= np.isfinite(ef) & np.isfinite(es)
    mf, ms_ = np.nanmedian(ef[lo:]), np.nanmedian(es[lo:])
    grind = base & (ef < mf) & (es >= ms_)
    r['W_grind'] = (float(sw[grind].mean() * 100) if grind.sum() >= MIN_CELL_BARS
                    else np.nan)
    r['n_grind'] = int(grind.sum())

    q = np.nanquantile(ef[lo:], [0.2, 0.4, 0.6, 0.8])
    r['W_quintile'] = []
    for b in range(5):
        band = (ef >= (q[b - 1] if b else -1)) & (ef < (q[b] if b < 4 else 2))
        m = base & band
        r['W_quintile'].append(float(sw[m].mean() * 100) if m.sum() >= 100
                                else np.nan)

    Xoos = r['feat'].values[lo:]
    Xoos = Xoos[np.isfinite(Xoos).all(axis=1)]
    r['cond'], r['eff_dims'] = feature_conditioning(Xoos)
    r['eff_share'] = r['eff_dims'] / Xoos.shape[1]

print(f'grind-cell coherence + conditioning computed for {len(FEATSET_RUNS)} '
      'runs (OOS span, seed set 0, EFF_FAST=9 bars / EFF_SLOW=20 days).')

## 8. Results — nested vs disjoint, side by side per instrument, never pooled

In [ ]:
# ===========================================================================
# MAIN TABLE. Every run appears. Nothing dropped for looking bad.
# ===========================================================================
def guard_str(r):
    return ''.join(('.' if r['guards'][k][0] else 'X') for k in ('G1', 'G2', 'G3', 'G4'))


def get(inst, design):
    return next(r for r in FEATSET_RUNS if r['inst'] == inst and r['design'] == design)


print('=' * 150)
print('FEATURE-SET COMPARISON -- 7 instruments x {nested, disjoint}, 1h, CONTEXT_DAYS=12 for both')
print('=' * 150)
print(f"{'instrument':<10}{'design':<10}{'S':>6}{'L':>7}{'W':>7}{'guards':>8}"
      f"{'W_grind':>9}{'n_grind':>9}{'cond#':>8}{'eff_dims':>9}{'OOS t':>8}")
print('-' * 150)
for d in ALL_INST:
    for design in DESIGNS:
        r = get(d['name'], design)
        print(f"{d['name']:<10}{design:<10}{r['S']:>6.2f}{r['L']:>7.1f}"
              f"{r['W']:>7.2f}{guard_str(r):>8}{r['W_grind']:>9.2f}"
              f"{r['n_grind']:>9}{r['cond']:>8.2f}{r['eff_dims']:>9.2f}"
              f"{r['OOS_T']:>8.2f}")
    print('-' * 150)

_gf = [r for r in FEATSET_RUNS if not r['guards_ok']]
print(f'guards: {len(FEATSET_RUNS) - len(_gf)} of {len(FEATSET_RUNS)} runs pass '
      'all four'
      + ('' if not _gf else '  FAILING: '
         + ', '.join(f"{r['inst']}|{r['design']}" for r in _gf)))

In [ ]:
# ===========================================================================
# QUINTILE TABLE (F2) -- W by eff_fast quintile, nested vs disjoint, per
# instrument and pooled.
# ===========================================================================
print('=' * 130)
print('WHIPSAW W BY eff_fast QUINTILE (Q1=choppiest bars by the 9-bar filter, '
      'Q5=most efficient/violent)')
print('=' * 130)
print(f"{'instrument':<10}{'design':<10}" + ''.join(f"{'Q' + str(i):>9}" for i in range(1, 6)))
print('-' * 130)
POOLED_Q = {design: [[] for _ in range(5)] for design in DESIGNS}
for d in ALL_INST:
    for design in DESIGNS:
        r = get(d['name'], design)
        print(f"{d['name']:<10}{design:<10}"
              + ''.join(f"{v:>9.2f}" if np.isfinite(v) else f"{'n/a':>9}"
                        for v in r['W_quintile']))
        for i, v in enumerate(r['W_quintile']):
            if np.isfinite(v):
                POOLED_Q[design][i].append(v)
print('-' * 130)
POOLED_Q_MEAN = {design: [float(np.mean(q)) if q else np.nan for q in POOLED_Q[design]]
                 for design in DESIGNS}
for design in DESIGNS:
    print(f"{'POOLED MEAN':<10}{design:<10}"
          + ''.join(f"{v:>9.2f}" for v in POOLED_Q_MEAN[design]))

## 9. Figures — look at these before trusting any table above

In [ ]:
# ===========================================================================
# FIGURE 1 -- regime-on-price, nested vs disjoint, OOS span, 3 instruments
# that showed the barcode most clearly in the earlier diagnostic probes
# (AUD/USD, USD/CAD) plus EUR/USD as a violent-move sanity check.
# ===========================================================================
import matplotlib.pyplot as plt

FIG1_INST = ['AUD/USD', 'USD/CAD', 'EUR/USD']
fig, axes = plt.subplots(len(FIG1_INST), 2, figsize=(15, 11), sharex='row')
for i, inst in enumerate(FIG1_INST):
    for j, design in enumerate(DESIGNS):
        ax = axes[i][j]
        r = get(inst, design)
        lo = r['n_fit']
        dates, px, lab = r['dates'][lo:], r['close_arr'][lo:], np.asarray(r['labels'][0], object)[lo:]
        shade_regimes(ax, pd.Series(lab, index=dates), alpha=0.55)
        ax.plot(dates, px, color='black', lw=0.7)
        ax.set_xlim(dates[0], dates[-1])
        ax.margins(y=0.05)
        occ = {k: float(np.mean(lab == k)) * 100 for k in REGIME_LABELS}
        ax.set_title(f"{inst}  [{design}]   W {r['W']:.1f}/100   "
                     f"W_grind {r['W_grind']:.1f}/100 (n={r['n_grind']})",
                     fontsize=9.5)
        ax.tick_params(labelsize=7.5)
        if j == 0:
            ax.set_ylabel('price', fontsize=8)
handles = [plt.matplotlib.patches.Patch(color=REGIME_COLORS[k], alpha=0.7, label=k)
           for k in REGIME_LABELS]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, 0.005))
fig.suptitle('NESTED (shipped, left) vs DISJOINT (right) -- out-of-sample span '
             'only, CONTEXT_DAYS=12 both, one seed set (for drawing)',
             fontsize=12, fontweight='bold')
fig.tight_layout(rect=[0, 0.035, 1, 0.955])
plt.show()

In [ ]:
# ===========================================================================
# FIGURE 2 -- F1/F2 evidence: quintile W curve (line) + grind-cell W (bars).
# ===========================================================================
fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 5.5))

for design, style in DESIGNS.items():
    axL.plot(range(1, 6), POOLED_Q_MEAN[design], marker='o',
             label=design, lw=2.2)
axL.set_xticks(range(1, 6))
axL.set_xticklabels([f'Q{i}' for i in range(1, 6)])
axL.set_xlabel('eff_fast quintile (Q1=choppiest -> Q5=most violent)')
axL.set_ylabel('mean whipsaw W (switches / 100 bars)')
axL.set_title('F2: whipsaw by move-quality quintile, pooled over 7 instruments')
axL.legend()
axL.grid(alpha=0.3)

_x = np.arange(len(ALL_INST))
_w = 0.36
for k, design in enumerate(DESIGNS):
    vals = [get(d['name'], design)['W_grind'] for d in ALL_INST]
    axR.bar(_x + (k - 0.5) * _w, vals, width=_w, label=design)
axR.set_xticks(_x)
axR.set_xticklabels([d['name'] for d in ALL_INST], rotation=30, ha='right')
axR.set_ylabel('W_grind (switches / 100 bars, "steady trend with wiggles" cell)')
axR.set_title('F1: whipsaw in the exact cell the user flagged by eye')
axR.legend()
axR.grid(alpha=0.3, axis='y')

fig.suptitle('Does the disjoint feature set reduce whipsaw where the user '
             'said it should, without breaking where it already worked?',
             fontsize=12, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

## 10. Grading F1–F7 — against the rules frozen in the protocol, before any number existed

In [ ]:
# ===========================================================================
# GRADING -- verbatim against FX_FEATURESET_PROTOCOL.md. No threshold moved.
# ===========================================================================
print('=' * 108)
print('GRADING THE PRE-REGISTERED PREDICTIONS')
print('=' * 108)

# ---- F1: primary, the user's visual complaint --------------------------
_f1_pairs = [(d['name'], get(d['name'], 'nested')['W_grind'],
             get(d['name'], 'disjoint')['W_grind']) for d in ALL_INST]
_f1_def = [(n, wn, wd) for n, wn, wd in _f1_pairs if np.isfinite(wn) and np.isfinite(wd)]
_f1_wins = [n for n, wn, wd in _f1_def if wd < wn]
_f1_pool_nested = float(np.mean([wn for _, wn, _ in _f1_def])) if _f1_def else np.nan
_f1_pool_disjoint = float(np.mean([wd for _, _, wd in _f1_def])) if _f1_def else np.nan
print(f"\nF1  grind-cell W, per instrument (nested -> disjoint):")
for n, wn, wd in _f1_pairs:
    tag = ('disjoint LOWER' if (np.isfinite(wn) and np.isfinite(wd) and wd < wn)
           else ('disjoint HIGHER/EQUAL' if np.isfinite(wn) and np.isfinite(wd)
                 else 'UNDEFINED (< min cell bars)'))
    print(f"    {n:<10} {wn:>7.2f} -> {wd:>7.2f}   {tag}")
_f1_majority = len(_f1_wins) >= 4
_f1_pooled = np.isfinite(_f1_pool_disjoint) and _f1_pool_disjoint < _f1_pool_nested
print(f"    pooled mean: {_f1_pool_nested:.2f} -> {_f1_pool_disjoint:.2f}   "
      f"majority: {len(_f1_wins)} of {len(_f1_def)} defined")
if _f1_majority and _f1_pooled:
    print("F1  CONFIRMED")
elif _f1_pooled or _f1_majority:
    print(f"F1  SPLIT (not a confirmation) -- pooled {'moved' if _f1_pooled else 'did not move'} "
          f"the right way, majority {'held' if _f1_majority else 'did not hold'}")
else:
    print("F1  REFUTED")

# ---- F2: do-no-harm on the violent-move axis ----------------------------
_q_dj = POOLED_Q_MEAN['disjoint']
_f2_monotone = all(_q_dj[i] >= _q_dj[i + 1] for i in range(4)
                    if np.isfinite(_q_dj[i]) and np.isfinite(_q_dj[i + 1]))
print(f"\nF2  disjoint pooled W by quintile: "
      f"{['%.2f' % v if np.isfinite(v) else 'n/a' for v in _q_dj]}")
print(f"F2  {'CONFIRMED' if _f2_monotone else 'REFUTED'}")

# ---- F3: conditioning, measured live ------------------------------------
_f3_hits = 0
print(f"\nF3  conditioning per instrument (nested -> disjoint):")
for d in ALL_INST:
    rn, rd = get(d['name'], 'nested'), get(d['name'], 'disjoint')
    hit = rd['cond'] < rn['cond'] and rd['eff_share'] > rn['eff_share']
    _f3_hits += int(hit)
    print(f"    {d['name']:<10} cond {rn['cond']:>7.2f}->{rd['cond']:>7.2f}   "
          f"eff_share {rn['eff_share']:.2f}->{rd['eff_share']:.2f}   "
          f"{'BOTH IMPROVE' if hit else 'NO'}")
print(f"F3  {_f3_hits} of {len(ALL_INST)} instruments both-improve -- "
      f"{'CONFIRMED' if _f3_hits >= 4 else 'REFUTED'}")

# ---- F4: arithmetic, already shown ---------------------------------------
print(f"\nF4  n_params: nested {_p_nested:.0f} > disjoint {_p_disjoint:.0f}   "
      f"CONFIRMED")

# ---- F5: do-no-harm, structural -------------------------------------------
_nested_pass = sum(1 for d in ALL_INST if get(d['name'], 'nested')['guards_ok'])
_disjoint_pass = sum(1 for d in ALL_INST if get(d['name'], 'disjoint')['guards_ok'])
print(f"\nF5  guards pass: nested {_nested_pass}/{len(ALL_INST)}   "
      f"disjoint {_disjoint_pass}/{len(ALL_INST)}   "
      f"{'CONFIRMED' if _disjoint_pass >= _nested_pass else 'REFUTED'}")

# ---- F6: bistability / seed-stability -------------------------------------
_s_nested = float(np.mean([get(d['name'], 'nested')['S'] for d in ALL_INST]))
_s_disjoint = float(np.mean([get(d['name'], 'disjoint')['S'] for d in ALL_INST]))
print(f"\nF6  mean S: nested {_s_nested:.3f}   disjoint {_s_disjoint:.3f}   "
      f"{'CONFIRMED' if _s_disjoint > _s_nested else 'REFUTED'}")

# ---- F7: honesty check, forward-return skill -------------------------------
_dt = [get(d['name'], 'disjoint')['OOS_T'] - get(d['name'], 'nested')['OOS_T']
       for d in ALL_INST
       if np.isfinite(get(d['name'], 'disjoint')['OOS_T'])
       and np.isfinite(get(d['name'], 'nested')['OOS_T'])]
_mean_dt = float(np.mean(_dt)) if _dt else np.nan
print(f"\nF7  mean OOS HAC-t (disjoint - nested): {_mean_dt:+.3f}   "
      f"(n={len(_dt)} of {len(ALL_INST)} defined)")
print(f"F7  {'CONFIRMED' if np.isfinite(_mean_dt) and _mean_dt >= -0.20 else 'REFUTED'}"
      "  -- NOT the goal; does not gate F1-F6 either way.")
print()
print('Reminder: none of the above is a return, a Sharpe, or a tradeability '
      'claim. Data ends 2022-03-04.')

## 11. Hand-off — what a decision to ship this would actually change

In [ ]:
# ===========================================================================
# Nothing here is shipped by running this notebook. This block states what
# WOULD change in the master if the grading above supports adopting it.
# ===========================================================================
print('=' * 100)
print('IF ADOPTED, THIS IS THE DIFF FROM THE SHIPPED MASTER (build_master_notebook_v2.py):')
print('=' * 100)
print(f"  FEATURE_COLS      {FEATURE_COLS_BASELINE}")
print(f"                 -> {DISJOINT_FEATURE_COLS}")
print(f"  FEATURE_SIGN/MAG  updated for the 4 renamed momentum features (see cell 2)")
print(f"  DIRECTION_EXCLUDE ('vol_2h','vol_expansion') -> ('vol_2h',)")
print(f"  BAR_DIR_FEATURES  ('ret_2h','mom_1d','mom_3d','mom_5d','dist_ma')")
print(f"                 -> ('d0_1','d1_5','d5_20','d20_60','dist_ma')")
print(f"  build_features()  replaced by build_features_disjoint() (this notebook, cell 2)")
print(f"  TREND_FEATURE     UNCHANGED ('mom_3d') -- INTENSITY axis untouched")
print(f"  n_params(N=5,diag) {_p_nested:.0f} -> {_p_disjoint:.0f}")
print()
print('NOT changed by this notebook: N_STATES, COVARIANCE, BAR_DIR_WEIGHT, '
      'ENSEMBLE_K, CONF_L, CONFIRM_BARS, Z_HI/EFF_HI bands, CONTEXT_DAYS, '
      'TRAIN_FRACTION, N_FOLDS, INTENSITY_MODE, ESCALATION_DURING_HOLD, '
      'DIRECTION_MODE.')
print()
print('This notebook does not modify build_master_notebook_v2.py. Adoption is '
      'a separate, explicit step.')